# 10: What actually drives complementarity? (EXPLORATORY)

Entirely exploratory. The confirmatory read (lock v3) is already banked; nothing here carries
a verdict, and no result from this notebook may be described as pre-specified.

Overall error correlation (rho) failed to predict fresh-set advantage. Three candidate reasons,
each tested here:

1. **Wrong errors.** Rho was measured over all errors, which at 25% referral are dominated by
   false positives and grade-2 misses. The outcome depends on a few hundred severe eyes.
   Fix: dependence measured **on severe cases only**.
2. **Marginals dominate.** Under independence, the joint miss rate is m_A x m_B, so the payoff
   from decorrelation shrinks as models improve. Fix: **decompose** the union advantage into
   what independence alone predicts and what observed dependence adds.
3. **Uncorrelated is not useful.** A partner must rank the severe cases the first model missed
   *highly*, not merely differently. Fix: **rank complementarity**, where the partner places
   the severe eyes its teammate missed.

Every candidate predictor is measured on **validation** and tested against **fresh-set**
advantage, so the comparison is out of sample, exactly as the rho test was.

Also reported: matched-workload advantage versus union advantage, since at matched workload
the OR-gate cannot refer the full union and truncation may itself limit the benefit.

CPU runtime, about 15 minutes. Run cells 1-6.

In [1]:
# Cell 1: load validation and fresh predictions
import io, os, json, itertools, datetime, time
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from google.colab import auth
auth.authenticate_user()
from googleapiclient.discovery import build as gbuild
from googleapiclient.http import MediaIoBaseDownload, MediaFileUpload
svc = gbuild('drive', 'v3')
F_PREDICTIONS = '1nCloJ5o9veZb2xJL_2EjdVHQXN31jy4w'; F_RESULTS = '1-HT4ONiwIfuBtC19s7QkDd8Euklh0Xon'
F_FIGURES = '18eyHRWuqFcUrYiL8nPtpIalV7CreLkMK'

def dl(fid, path):
    with io.FileIO(path, 'wb') as fh:
        d = MediaIoBaseDownload(fh, svc.files().get_media(fileId=fid)); done = False
        while not done: _, done = d.next_chunk()
def find_id(n, f):
    r = svc.files().list(q=f"'{f}' in parents and name='{n}' and trashed=false", fields='files(id)').execute().get('files', [])
    return r[0]['id'] if r else None
def up(p, f, m):
    svc.files().create(body={'name': os.path.basename(p), 'parents': [f]},
                       media_body=MediaFileUpload(p, mimetype=m), fields='id').execute(); print('  uploaded', os.path.basename(p))

ARCHS = ['densenet121_binary','densenet121_5class','efficientnet_b3_binary','efficientnet_b3_5class',
         'resnet50_binary','resnet50_5class','vit_base_binary','vit_base_5class',
         'retfound_binary','retfound_5class','retfound_adversarial']
SEEDS = [0,1,2]; PAIRS = list(itertools.combinations(ARCHS, 2)); COMBOS = [(i,j) for i in SEEDS for j in SEEDS]
def tag(k): return '' if k == 0 else f'_seed{k}'
BURDEN = 0.25

def load(prefix, split):
    P, ref = {}, None
    for a in ARCHS:
        for k in SEEDS:
            fn = f'{prefix}{a}{tag(k)}_v2.csv'
            if not os.path.exists(fn): dl(find_id(fn, F_PREDICTIONS), fn)
            d = pd.read_csv(fn); assert set(d['split']) == {split}
            if ref is None: ref = d[['image_id','true_grade','true_label']].reset_index(drop=True)
            P[(a,k)] = d['predicted_probability'].to_numpy(np.float64)
    return P, ref

PV, RV = load('eyepacs_predictions_val_', 'val')
PF, RF = load('eyepacs_predictions_fresh_', 'fresh')
GV, GF = RV['true_grade'].to_numpy(int), RF['true_grade'].to_numpy(int)
SEV_V, SEV_F = GV >= 3, GF >= 3
NV, NF = len(GV), len(GF)
RVn, RFn = int(round(BURDEN*NV)), int(round(BURDEN*NF))
print(f'validation {NV} images ({SEV_V.sum()} severe) | fresh {NF} images ({SEV_F.sum()} severe)')

validation 2900 images (132 severe) | fresh 15794 images (712 severe)


In [2]:
# Cell 2: referral sets and the three candidate quantities, on either dataset
def referred(p, R):
    m = np.zeros(len(p), bool); m[np.argsort(-p, kind='stable')[:R]] = True; return m

def quantities(P, SEV, R, N):
    """Per pair and seed combination: marginal severe-miss rates, joint (union) miss rate,
    the independence benchmark, matched-workload advantage, and rank complementarity."""
    REF = {key: referred(p, R) for key, p in P.items()}
    RK  = {key: rankdata(p) / N for key, p in P.items()}
    nsev = SEV.sum(); rows = []
    for a, b in PAIRS:
        for i, j in COMBOS:
            ma = (~REF[(a,i)]) & SEV; mb = (~REF[(b,j)]) & SEV        # severe eyes each misses
            m_a, m_b = ma.sum()/nsev, mb.sum()/nsev
            both = (ma & mb).sum()/nsev
            indep = m_a * m_b
            # matched workload OR: top-R by max rank (the locked primary definition)
            or_ref = referred(np.maximum(RK[(a,i)], RK[(b,j)]), R)
            m_or = ((~or_ref) & SEV).sum()/nsev
            worse, better = (ma, mb) if m_a >= m_b else (mb, ma)      # 'better' = fewer severe misses
            partner_rank = RK[(b,j)] if m_a >= m_b else RK[(a,i)]     # partner of the worse model
            miss_better = better
            partner_of_better = RK[(b,j)] if m_a < m_b else RK[(a,i)]
            rescued = (miss_better & referred(partner_of_better, R)).sum()
            rows.append({'model_a': a, 'model_b': b, 'seed_a': i, 'seed_b': j,
                         'm_a': m_a, 'm_b': m_b, 'm_min': min(m_a, m_b), 'm_both_union': both,
                         'm_indep': indep, 'dependence_excess': both - indep,
                         'adv_union': min(m_a, m_b) - both,
                         'adv_union_if_independent': min(m_a, m_b) - indep,
                         'adv_matched': min(m_a, m_b) - m_or,
                         'partner_mean_rank_of_missed': float(partner_of_better[miss_better].mean()) if miss_better.any() else np.nan,
                         'rescued_frac': rescued / max(1, miss_better.sum())})
    return pd.DataFrame(rows)

t0 = time.time()
QV = quantities(PV, SEV_V, RVn, NV); QF = quantities(PF, SEV_F, RFn, NF)
print(f'computed in {time.time()-t0:.0f}s | rows {len(QV)} each')
print(QF[['m_min','m_both_union','m_indep','dependence_excess','adv_union','adv_matched','rescued_frac']].describe().round(4).to_string())

computed in 2s | rows 495 each
          m_min  m_both_union   m_indep  dependence_excess  adv_union  adv_matched  rescued_frac
count  495.0000      495.0000  495.0000           495.0000   495.0000     495.0000      495.0000
mean     0.0523        0.0299    0.0037             0.0262     0.0224       0.0067        0.4206
std      0.0105        0.0065    0.0011             0.0058     0.0080       0.0071        0.1041
min      0.0351        0.0183    0.0014             0.0148     0.0028      -0.0126        0.0690
25%      0.0449        0.0253    0.0028             0.0219     0.0169       0.0014        0.3636
50%      0.0506        0.0281    0.0035             0.0252     0.0211       0.0070        0.4242
75%      0.0618        0.0337    0.0044             0.0302     0.0281       0.0112        0.4860
max      0.0843        0.0548    0.0072             0.0486     0.0449       0.0267        0.6818


In [3]:
# Cell 3: does dependence add anything beyond the marginals? (fresh set)
g = QF.groupby(['model_a','model_b']).mean(numeric_only=True)
tot = g['adv_union'].mean(); indep_part = g['adv_union_if_independent'].mean()
dep_part = tot - indep_part
DECOMP = {
  'mean_union_advantage': round(float(tot), 4),
  'predicted_by_independence_alone': round(float(indep_part), 4),
  'added_by_observed_dependence': round(float(dep_part), 4),
  'share_from_dependence': round(float(dep_part / tot), 3) if tot else None,
  'mean_joint_miss_rate': round(float(g['m_both_union'].mean()), 4),
  'mean_independence_benchmark': round(float(g['m_indep'].mean()), 4),
  'pairs_more_dependent_than_independent_of_55': int((g['dependence_excess'] > 0).sum()),
  'mean_matched_advantage': round(float(g['adv_matched'].mean()), 4),
  'matched_as_share_of_union': round(float(g['adv_matched'].mean() / tot), 3) if tot else None,
}
print(json.dumps(DECOMP, indent=2))
print('\nReading: if "added_by_observed_dependence" is near zero, the ensemble benefit is a')
print('consequence of marginal severe-case performance, not of error decorrelation.')
print('A positive dependence_excess means models miss the SAME severe eyes more often than chance.')

{
  "mean_union_advantage": 0.0224,
  "predicted_by_independence_alone": 0.0486,
  "added_by_observed_dependence": -0.0262,
  "share_from_dependence": -1.172,
  "mean_joint_miss_rate": 0.0299,
  "mean_independence_benchmark": 0.0037,
  "pairs_more_dependent_than_independent_of_55": 55,
  "mean_matched_advantage": 0.0067,
  "matched_as_share_of_union": 0.297
}

Reading: if "added_by_observed_dependence" is near zero, the ensemble benefit is a
consequence of marginal severe-case performance, not of error decorrelation.
A positive dependence_excess means models miss the SAME severe eyes more often than chance.


In [4]:
# Cell 4: which validation-measured quantity predicts fresh advantage? (out of sample)
gv = QV.groupby(['model_a','model_b']).mean(numeric_only=True)
gf = QF.groupby(['model_a','model_b']).mean(numeric_only=True)
idx = [(a, b) for a, b in PAIRS]
lockv2 = 'lock_v2.json'
if not os.path.exists(lockv2): dl(find_id('locked_config.json', F_RESULTS), lockv2)
RHO = {tuple(k.split('|')): v for k, v in json.load(open(lockv2))['rho']['validation'].items()}

y = np.array([gf.loc[p, 'adv_matched'] for p in idx])
CANDS = {
  'overall_rho_all_errors (locked, failed)': np.array([RHO[p] for p in idx]),
  'severe_dependence_excess': np.array([gv.loc[p, 'dependence_excess'] for p in idx]),
  'min_marginal_severe_miss_rate': np.array([gv.loc[p, 'm_min'] for p in idx]),
  'mean_marginal_severe_miss_rate': np.array([(gv.loc[p, 'm_a'] + gv.loc[p, 'm_b']) / 2 for p in idx]),
  'rank_complementarity (rescued_frac)': np.array([gv.loc[p, 'rescued_frac'] for p in idx]),
  'partner_mean_rank_of_missed': np.array([gv.loc[p, 'partner_mean_rank_of_missed'] for p in idx]),
}
rng = np.random.default_rng(20260922)
def model_level_ci(x, y, B=2000):
    out = []
    for _ in range(B):
        s = rng.choice(ARCHS, len(ARCHS), replace=True)
        ps = [(u, v) if (u, v) in gf.index else (v, u) for u, v in itertools.combinations(s, 2) if u != v]
        if len(set(ps)) < 5: continue
        k = [idx.index(p) for p in ps]
        out.append(spearmanr(x[k], y[k]).correlation)
    return [round(float(v), 3) for v in np.nanpercentile(out, [2.5, 97.5])]

PRED = []
for name, x in CANDS.items():
    ok = ~np.isnan(x)
    r = spearmanr(x[ok], y[ok]).correlation
    PRED.append({'predictor (validation)': name, 'spearman_vs_fresh_matched_advantage': round(float(r), 3),
                 'model_level_95ci': model_level_ci(x, y)})
PREDF = pd.DataFrame(PRED)
print(PREDF.to_string(index=False))
print('\nAll out of sample: predictor from validation, outcome from the fresh set.')

                 predictor (validation)  spearman_vs_fresh_matched_advantage model_level_95ci
overall_rho_all_errors (locked, failed)                                0.255  [-0.401, 0.694]
               severe_dependence_excess                                0.168  [-0.294, 0.579]
          min_marginal_severe_miss_rate                                0.391  [-0.209, 0.812]
         mean_marginal_severe_miss_rate                                0.353  [-0.142, 0.666]
    rank_complementarity (rescued_frac)                               -0.007  [-0.425, 0.382]
            partner_mean_rank_of_missed                               -0.006  [-0.458, 0.387]

All out of sample: predictor from validation, outcome from the fresh set.


In [5]:
# Cell 5: where does a partner place the severe eyes its teammate missed?
sub = QF.dropna(subset=['partner_mean_rank_of_missed'])
RANKS = {'mean_partner_rank_of_missed_severe': round(float(sub['partner_mean_rank_of_missed'].mean()), 4),
         'reference_rank_if_uninformative': round(1 - BURDEN / 2, 4),
         'mean_rescued_fraction': round(float(sub['rescued_frac'].mean()), 4),
         'rescued_fraction_iqr': [round(float(x), 4) for x in sub['rescued_frac'].quantile([0.25, 0.75])]}
print(json.dumps(RANKS, indent=2))
print('\nA partner ranking missed severe eyes at ~0.75+ is placing them near its own referral')
print('threshold; the rescued fraction is how many it would actually refer at the same workload.')

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4.4))
ax[0].scatter(CANDS['overall_rho_all_errors (locked, failed)'], y, s=18)
ax[0].set_xlabel('validation rho (all errors)'); ax[0].set_title('locked predictor (failed)')
ax[1].scatter(CANDS['severe_dependence_excess'], y, s=18, color='C1')
ax[1].set_xlabel('validation severe-case dependence excess'); ax[1].set_title('severe-specific dependence')
ax[2].scatter(CANDS['min_marginal_severe_miss_rate'], y, s=18, color='C2')
ax[2].set_xlabel('validation min marginal severe-miss rate'); ax[2].set_title('marginal performance')
for a_ in ax: a_.set_ylabel('fresh matched-workload advantage'); a_.axhline(0, color='grey', lw=0.8)
fig.suptitle('What predicts ensemble complementarity? (exploratory, out of sample)')
fig.tight_layout(); fig.savefig('complementarity_predictors_v3.png', dpi=200)
print('figure written')

{
  "mean_partner_rank_of_missed_severe": 0.6472,
  "reference_rank_if_uninformative": 0.875,
  "mean_rescued_fraction": 0.4206,
  "rescued_fraction_iqr": [
    0.3636,
    0.486
  ]
}

A partner ranking missed severe eyes at ~0.75+ is placing them near its own referral
threshold; the rescued fraction is how many it would actually refer at the same workload.
figure written


In [6]:
# Cell 6: save
OUT = {'generated_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
       'status': 'EXPLORATORY: generated after the confirmatory read; not pre-specified',
       'burden': BURDEN, 'severe_definition': 'grade >= 3',
       'decomposition_fresh': DECOMP, 'predictors_out_of_sample': PRED, 'rank_complementarity': RANKS}
with open('complementarity_mechanism_v3.json', 'w') as f: json.dump(OUT, f, indent=2, default=str)
QF.to_csv('complementarity_quantities_fresh_v3.csv', index=False)
QV.to_csv('complementarity_quantities_val_v3.csv', index=False)
for fn, m in (('complementarity_mechanism_v3.json','application/json'),
              ('complementarity_quantities_fresh_v3.csv','text/csv'),
              ('complementarity_quantities_val_v3.csv','text/csv')):
    up(fn, F_RESULTS, m)
up('complementarity_predictors_v3.png', F_FIGURES, 'image/png')
print(json.dumps(OUT['decomposition_fresh'], indent=2))

  uploaded complementarity_mechanism_v3.json
  uploaded complementarity_quantities_fresh_v3.csv
  uploaded complementarity_quantities_val_v3.csv
  uploaded complementarity_predictors_v3.png
{
  "mean_union_advantage": 0.0224,
  "predicted_by_independence_alone": 0.0486,
  "added_by_observed_dependence": -0.0262,
  "share_from_dependence": -1.172,
  "mean_joint_miss_rate": 0.0299,
  "mean_independence_benchmark": 0.0037,
  "pairs_more_dependent_than_independent_of_55": 55,
  "mean_matched_advantage": 0.0067,
  "matched_as_share_of_union": 0.297
}
